## Typicality rating experiment — LLaMA 3.1 8B Instruct

In [1]:
import os
# Must be set BEFORE importing transformers, or it will still probe TensorFlow
#os.environ["USE_TF"] = "0"
#os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

import gc
import re
import pandas as pd
import torch
from transformers import pipeline

#MODEL_NAME = "/data1/shared_models/models--meta-llama--Llama-3.1-8B-Instruct/"  # change to "meta-llama/Meta-Llama-3-8B-Instruct" if you need 3.0

2026-07-16 13:16:25.548475: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-16 13:16:25.561519: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784200585.577432    7081 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784200585.582219    7081 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784200585.594203    7081 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/home/muti/miniconda3/envs/testenv/lib/python3.11/site-packages/tensorflow/python/keras/engine/training_arrays_v1.py:37: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.6)
  from scipy.sparse import issparse  # pylint: disable=g-import-not-at-top


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
# Set the language to run this notebook for.
# Must match the suffix used in your CSV's instance_<LANGUAGE> column,
# e.g. "English", "German", "Spanish"
LANGUAGE = "German"

instance_col = f"instance_{LANGUAGE}"
norm_rating_col = f"norm_rating_{LANGUAGE}"


In [4]:
import torch, gc
torch.cuda.empty_cache()
gc.collect()

100

In [5]:
df = pd.read_csv('exp_1/combined_prototypes.csv')
df.head(5)
df = df[df[instance_col].notna() & (df[instance_col].astype(str).str.strip() != "")].reset_index(drop=True)

In [6]:
len(df)

181

In [7]:
def typicality_prompt(obj, category):
    return f"""
You are participating in a psychology experiment.

Rate how prototypical "{obj}" is as an example of the category "{category}".
Respond with ONLY one integer between 1 and 7, where 1 is not prototypical at all and 7 is very prototypical.
"""

In [8]:
def get_typicality(obj, category):
    prompt = typicality_prompt(obj, category)
    messages = [
        {"role": "user", "content": prompt},
    ]
    output = pipe(
        messages,
        max_new_tokens=5,
        do_sample=False,
    )
    response = output[0]["generated_text"][-1]["content"]
    match = re.search(r"[1-7]", response)
    if match:
        return int(match.group())
    else:
        print(f"Could not parse response for '{obj}' ({category}): {response!r}")
        return None


In [9]:
ls /data1/shared_models/models--google--gemma-3-12b-it/snapshots/

96b6f1eccf38110c56df3a15bffe176da04bfd80/


In [10]:
from transformers import pipeline
MODEL_NAME = "/data1/shared_models/models--google--gemma-3-12b-it/snapshots/96b6f1eccf38110c56df3a15bffe176da04bfd80//"
#MODEL_NAME = "/data1/shared_models/models--meta-llama--Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659/"

pipe = pipeline(
    "text-generation",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.
Device set to use cuda:0


In [11]:
# Reset column in case a previous buggy run stored prompt text instead of scores
df["llama_typicality"] = pd.to_numeric(df["llama_typicality"], errors="coerce") if "llama_typicality" in df.columns else None

output_path = f"combined_prototypes_{LANGUAGE}.csv"

for idx, row in df.iterrows():
    if pd.notna(df.at[idx, "llama_typicality"]):
        continue  # already done, skip (useful on resume)
    score = get_typicality(row[instance_col], row["category"])
    df.at[idx, "llama_typicality"] = score
    df.to_csv(output_path, index=False)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [12]:
df.to_csv(f"gemma_{LANGUAGE.lower()}_likert.csv", index=False)
